# Native CLM Architecture Lab — Phase A

This is the single long-lived architecture-search notebook. C0/T1 seed 91001 are frozen anchors; Phase A spends the same 10M-token budget on four orthogonal CLM candidates and does not retrain T1.


In [ ]:
BRANCH = "research/native-clm-lab"
PROFILE = "baseline"
SEED = 91001
SEARCH_MODELS = [
    "C1-residual-gated-update",
    "C2-shared-wide-phase",
    "C3-progressive-receptive-field",
    "C4-sparse-communication",
]
RUN_SEARCH = True
ALLOW_CPU = False
PUSH_DEV_RESULT = True
assert SEED == 91001, "Phase A must not consume additional seeds"


## Environment bootstrap

Large caches/checkpoints stay under `/kaggle/working/native-clm`. On two GPUs the sweep runs two complete candidates concurrently.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

def run_checked(cmd, **kwargs):
    return subprocess.run(cmd, check=True, text=True, **kwargs)

repo_candidates = [Path.cwd(), Path("/kaggle/working/mini-cells")]
REPO_ROOT = next((p for p in repo_candidates if (p / "research/native-clm/run_native_clm_search.py").is_file()), None)
if REPO_ROOT is None:
    if not Path("/kaggle/working").exists():
        raise RuntimeError("Run inside the repository or on Kaggle")
    REPO_ROOT = Path("/kaggle/working/mini-cells")
    run_checked(["git","clone","--depth","1","--branch",BRANCH,"https://github.com/ArcheLabs/mini-cells.git",str(REPO_ROOT)])
run_checked([sys.executable,"-m","pip","install","-q","-e",str(REPO_ROOT)+"[lm]"])
WORK_ROOT = Path("/kaggle/working/native-clm") if Path("/kaggle/working").exists() else REPO_ROOT / ".native-clm-work"
CACHE_ROOT, OUTPUT_ROOT, HF_ROOT = WORK_ROOT/"cache", WORK_ROOT/"runs", WORK_ROOT/"huggingface"
for p in (CACHE_ROOT, OUTPUT_ROOT, HF_ROOT): p.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_ROOT))
os.environ.setdefault("HF_DATASETS_CACHE", str(HF_ROOT/"datasets"))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(HF_ROOT/"hub"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
print("repo:", REPO_ROOT); print("work:", WORK_ROOT)


In [ ]:
def read_secret(name):
    if os.environ.get(name): return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None
HF_TOKEN = read_secret("HF_TOKEN"); GITHUB_TOKEN = read_secret("GITHUB_TOKEN")
if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN
print({"HF_TOKEN_available": bool(HF_TOKEN), "GITHUB_TOKEN_available": bool(GITHUB_TOKEN)})


## Audit Phase-A candidates

Every candidate must be within 1% of frozen T1 parameters. C0/T1 are read from immutable development evidence.


In [ ]:
sys.path.insert(0, str(REPO_ROOT / "research/native-clm"))
import native_clm_runtime as base
from native_clm_candidates import VERIFIED_DATASET_REVISION, candidate_config, estimate_flops, parameter_summary
base.DATASET_REVISION = VERIFIED_DATASET_REVISION
params = parameter_summary()
for name in SEARCH_MODELS:
    assert params[name]["relative_error"] < 0.01
audit = {
    "dataset_revision": VERIFIED_DATASET_REVISION,
    "profile": PROFILE, "seed": SEED,
    "candidate_parameters": params,
    "candidate_configs": {n: candidate_config(n) for n in SEARCH_MODELS},
    "estimated_train_flops": {n: estimate_flops(n, base.PROFILES[PROFILE].target_tokens)["train_flops_estimate"] for n in SEARCH_MODELS},
}
print(json.dumps(audit, indent=2))


## Run Phase A

Default behavior runs all four candidates. With two GPUs this becomes two rounds: C1/C2, then C3/C4. Resume checkpoints are reused automatically.


In [ ]:
RUNNER = REPO_ROOT / "research/native-clm/run_native_clm_search.py"
cmd = [sys.executable, str(RUNNER), "sweep", "--models", *SEARCH_MODELS, "--profile", PROFILE, "--seed", str(SEED), "--cache-root", str(CACHE_ROOT), "--output-root", str(OUTPUT_ROOT)]
if ALLOW_CPU: cmd.append("--allow-cpu")
print("launching Phase A:", SEARCH_MODELS)
if RUN_SEARCH: run_checked(cmd, cwd=REPO_ROOT)
else: print("RUN_SEARCH=False; existing outputs only")


## Architecture leaderboard

Primary quality signals are 10M validation PPL and log-token validation-NLL AUC. FLOPs, throughput and VRAM remain separate columns.


In [ ]:
RUN_DIR = OUTPUT_ROOT / f"architecture-search-{PROFILE}-seed-{SEED}"
LEADERBOARD = RUN_DIR / "architecture-leaderboard.json"
if not LEADERBOARD.is_file(): raise FileNotFoundError(LEADERBOARD)
rows = json.loads(LEADERBOARD.read_text(encoding="utf-8"))
try:
    import pandas as pd
    display(pd.DataFrame(rows).sort_values("validation_ppl_10m"))
except Exception:
    print(json.dumps(sorted(rows, key=lambda r:r["validation_ppl_10m"]), indent=2))


## Record development evidence

Only small summaries/curves are copied into Git. No optimizer state, token cache or model checkpoint is committed.


In [ ]:
record_dir = REPO_ROOT / "research/native-clm/results/dev" / f"architecture-search-{PROFILE}-seed-{SEED}"
record_dir.mkdir(parents=True, exist_ok=True)
for name in ("protocol.json","architecture-leaderboard.json","architecture-leaderboard.csv","search-summary.json"):
    src = RUN_DIR / name
    if src.is_file(): shutil.copy2(src, record_dir/name)
for model in SEARCH_MODELS:
    safe = model.split("-")[0]
    for src_name, dst_name in (("summary.json",f"{safe}-summary.json"),("checkpoints.csv",f"{safe}-checkpoints.csv")):
        src = RUN_DIR/model/src_name
        if src.is_file(): shutil.copy2(src, record_dir/dst_name)
ranked = sorted(rows, key=lambda r:r["validation_ppl_10m"])
readme = "# Native CLM Phase-A development evidence\n\n" + "\n".join(f"- {r['id']}: PPL={r['validation_ppl_10m']:.6f}, AUC-mean={r['log_token_nll_auc_mean']:.6f}, params={r['parameters']}" for r in ranked) + "\n\nSeed 91001 only; development evidence, not NCLM-001.\n"
(record_dir/"README.md").write_text(readme, encoding="utf-8")
print("development evidence:", record_dir)


In [ ]:
def sanitized_git_push(repo, record_dir, token):
    relative = record_dir.relative_to(repo)
    run_checked(["git","config","user.name","native-clm-kaggle"], cwd=repo)
    run_checked(["git","config","user.email","native-clm@users.noreply.github.com"], cwd=repo)
    run_checked(["git","add",str(relative)], cwd=repo)
    if subprocess.run(["git","diff","--cached","--quiet"], cwd=repo).returncode == 0:
        print("no new evidence to push"); return
    run_checked(["git","commit","-m",f"research: record Native CLM Phase-A seed {SEED}"], cwd=repo)
    if not token:
        print("GITHUB_TOKEN unavailable; committed locally only"); return
    remote = "https://x-access-token:" + token + "@github.com/ArcheLabs/mini-cells.git"
    run_checked(["git","push",remote,f"HEAD:{BRANCH}"], cwd=repo)
    print("pushed Phase-A evidence to", BRANCH)

if PUSH_DEV_RESULT: sanitized_git_push(REPO_ROOT, record_dir, GITHUB_TOKEN)
else: print("automatic GitHub push disabled")
